### HOMEWORK EXERCISE ASSIGNMENT
Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

#### Benefits:

    1. No API charges - open-source
    2. Data doesn't leave your box

#### Disadvantages:

    1. Significantly less power than Frontier Model

### Installing libraries

In [6]:
import os
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

### Setting system_prompt and user_prompt

In [7]:
system_prompt = """
You are a snarky assistant that analyzes the contents of the website, 
and provides a short, snarky, and humourous summary, ignoring text that might be
navigation related. Respond in markdown. Do not wrap the markdown in a code block -
respond just with the markdown.
"""

user_prompt = """
Here are the contents of the website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

Website content:
"""

### Fetching the contens of the website.

In [8]:
def fetch_website_contents(url):
    print(f"Fetching content from {url} using selenium...")
    driver = None

    try:
        print("1. Setting up chrome options....")
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("window-size=1200x600")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")

        print("2. Installing/Finding WebDriver....")
        driver_path = ChromeDriverManager().install()
        print(f"WebDriver is at: {driver_path}")
        service = Service(driver_path)

        print("3. Initializing WebDriver.....")
        driver = webdriver.Chrome(options=chrome_options, service= service)

        print("4. Setting page load timeout....")
        driver.set_page_load_timeout(10)

        print(f"5. Getting URL: {url}....")
        driver.get(url)

        print("6. Setting implicit wait....")
        driver.implicitly_wait(5)

        print("7. Finding body content....")
        body_content = driver.find_element(By.TAG_NAME, "body").text

        print("Content fetched successfully....")
        print(f"Fetched content length {len(body_content)}")

        return body_content
    
    except Exception as e:
        print(f"Error during selenium operation: {e}")
        if driver:
            body_content = driver.find_element(By.TAG_NAME, "body").text

            if body_content:
                print("Page timed out, but returing partial content.")
                return body_content
        return None

    finally:
        if driver:
            print("8. Shutting Down WebDriver....")
            driver.quit()
            print("WebDriver Shut Down...")

### Creating summarization function

In [21]:
def summarize_contents(url):
    from openai import OpenAI

    website = fetch_website_contents(url)

    if not website:
        return f"Sorry, I couldn't fetch the website."

    ollama_base_url = "http://localhost:11434/v1"
    ollama = OpenAI(base_url= ollama_base_url, api_key= "ollama")

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt + website}
    ]

    try:
        print("9. Calling Ollama....")

        response = ollama.chat.completions.create(
            model= "gemma3:270m",
            messages= messages
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error while calling Ollama {e}.....")
        return "Ollama failed to respond."   

In [22]:
print(summarize_contents("http://example.com"))

Fetching content from http://example.com using selenium...
1. Setting up chrome options....
2. Installing/Finding WebDriver....
WebDriver is at: C:\Users\salam\.wdm\drivers\chromedriver\win64\145.0.7632.117\chromedriver-win32/chromedriver.exe
3. Initializing WebDriver.....
4. Setting page load timeout....
5. Getting URL: http://example.com....
6. Setting implicit wait....
7. Finding body content....
Content fetched successfully....
Fetched content length 127
8. Shutting Down WebDriver....
WebDriver Shut Down...
9. Calling Ollama....
Here's a short summary of the website's content:



In [23]:
from IPython.display import Markdown, display

def display_summary(url):

    summary = summarize_contents(url)
    display(Markdown(summary))

In [24]:
display_summary("http://example.com")

Fetching content from http://example.com using selenium...
1. Setting up chrome options....
2. Installing/Finding WebDriver....
WebDriver is at: C:\Users\salam\.wdm\drivers\chromedriver\win64\145.0.7632.117\chromedriver-win32/chromedriver.exe
3. Initializing WebDriver.....
4. Setting page load timeout....
5. Getting URL: http://example.com....
6. Setting implicit wait....
7. Finding body content....
Content fetched successfully....
Fetched content length 127
8. Shutting Down WebDriver....
WebDriver Shut Down...
9. Calling Ollama....


```markdown
# Website Content Summary:

This website showcases tools and resources for various tasks. It contains a variety of examples, including basic scripting, data manipulation, and general documentation. The website prioritizes readability and clear explanations, avoiding overly complex language. It's intended for learners and professionals seeking to understand basic programming concepts.
```